In [12]:
"""
05 - Subgroup Tests
-------------------
Tests CAR differences across subgroups by
role positioning and cooperation type.

Test method
Paired t-test - Leading vs Partnering firm
Welch's t-test - Tech vs Non-Tech alliance

Data source : clean_event_data.csv
"""

import pandas as pd 
import numpy as np
from scipy import stats

df = pd.read_csv("clean_event_data.csv")

pairs = [
    ("Shuttle",    "InterServ"),
    ("Walsin",     "Teco (1)"),
    # SAA / SynPower excluded
    ("UMC",        "Chipbond"),
    ("TAC",        "ECPay"),
    ("GMTC",       "Soft-World"),
    ("Yao Sheng",  "JCP"),
    ("Foxconn",    "Teco (2)"),
]

techmap = {
    "Shuttle": 0, "InterServ": 0,
    "Walsin": 1, "Teco (1)": 1,
    "SAA": 1,
    "UMC": 1, "Chipbond": 1,
    "TAC": 0, "ECPay": 0,
    "GMTC": 0, "Soft-World": 0,
    "Yao Sheng": 1, "JCP": 1,
    "Foxconn": 1, "Teco (2)": 1,
}

windows = {
    "(-1,1)": (-1, 1),
    "(-5,5)": (-5, 5),
    "(-10,10)": (-10, 10)
}

#Paired t-test
for name, (start, end) in windows.items():
    temp = df[(df["Tau"] >= start) & (df["Tau"] <= end)]
    car = temp.groupby("firm")["AR"].sum()
    
    car_dif = np.array([
        car[lead] - car[partner]
        for lead, partner in pairs
    ])

    t_stat, p_val = stats.ttest_1samp(car_dif, popmean=0)

    print(f"Window {name}")
    print(f"Mean ΔCAR = {car_dif.mean():.4f} | t = {t_stat:.4f} | p = {p_val:.4f}")
    
print("\n")

#Welch's t-test
for name, (start, end) in windows.items():
    temp = df[(df["Tau"] >= start) & (df["Tau"] <= end)]
    car = temp.groupby("firm")["AR"].sum()
    
    car_tech = car[[f for f, v in techmap.items() if v==1 and f in car.index]].values
    car_ntech = car[[f for f, v in techmap.items() if v == 0 and f in car.index]].values

    t_stat, p_val = stats.ttest_ind(car_tech, car_ntech, equal_var=False)

    print(f"Window {name}")
    print(f"Mean CAR_Tech = {car_tech.mean():.4f} | Mean CAR_NTech = {car_ntech.mean():.4f} | t = {t_stat:.4f} | p = {p_val:.4f}")

Window (-1,1)
Mean ΔCAR = -0.3103 | t = -0.1275 | p = 0.9027
Window (-5,5)
Mean ΔCAR = 1.2140 | t = 0.2652 | p = 0.7997
Window (-10,10)
Mean ΔCAR = -2.6263 | t = -0.3480 | p = 0.7397


Window (-1,1)
Mean CAR_Tech = 2.0037 | Mean CAR_NTech = 2.3278 | t = -0.1226 | p = 0.9047
Window (-5,5)
Mean CAR_Tech = 2.5344 | Mean CAR_NTech = -6.1176 | t = 1.7544 | p = 0.1113
Window (-10,10)
Mean CAR_Tech = 1.3782 | Mean CAR_NTech = -5.3579 | t = 1.0321 | p = 0.3246
